# Param2Image: notebook training helper

Prefer local CLI: `pip install -e .` then `python scripts/train.py` with `./data/impact_dataset`.

This notebook discovers the repo root via `pyproject.toml` (or `P2I_REPO_ROOT`).
No machine-specific absolute paths are required.

## Step 1: Resolve repository root

In [ ]:
from pathlib import Path
import os

def resolve_repo_root() -> Path:
    """Find the Param2Image checkout without hardcoded machine/Colab paths."""
    env = os.environ.get("P2I_REPO_ROOT")
    if env:
        root = Path(env).expanduser().resolve()
        if (root / "pyproject.toml").exists():
            return root
        raise FileNotFoundError(f"P2I_REPO_ROOT is set but pyproject.toml missing: {root}")

    # Walk from cwd and common notebook layouts (cwd == repo or notebooks/)
    start = Path.cwd().resolve()
    candidates = [start, start.parent, *start.parents]
    for cand in candidates:
        if (cand / "pyproject.toml").exists() and (cand / "src" / "p2i").exists():
            return cand
    raise FileNotFoundError(
        "Could not locate repo root. cd into the clone, or set P2I_REPO_ROOT."
    )

REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("P2I_DATA_DIR", str(REPO_ROOT / "data" / "impact_dataset"))
print("REPO_ROOT =", REPO_ROOT)
print("P2I_DATA_DIR =", os.environ["P2I_DATA_DIR"])

## Step 2: Install package

In [ ]:
import sys
!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q -e .

import p2i
from p2i.paths import default_data_dir, default_generator_path, repo_root

print("p2i", p2i.__version__)
print("data_dir", default_data_dir())
print("generator", default_generator_path())

## Step 3: Check GPU

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Step 4: Train

In [ ]:
!python scripts/train.py

## Step 5: Generate a test case

In [ ]:
!python scripts/generate.py --velocity 200 --radius 5 --impact_x 0 --impact_y 0

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image
from p2i.paths import results_dir

pred_root = results_dir() / "predictions"
pred_dirs = sorted(pred_root.glob("sim_pred_*"))
if pred_dirs:
    img_path = pred_dirs[-1] / "generated_image.png"
    img = Image.open(img_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title("Generated impact image")
    plt.axis("off")
    plt.show()
else:
    print("No predictions found under", pred_root)